# Connection-Density Trade-offs — Interactive Demo

**What this notebook does:**
1. Loads the real experiment results from `results/raw_results.json` (no re-run needed)
2. Reproduces every plot with full annotations
3. Walks through the P1–P4 verdict logic step-by-step
4. Lets you explore parameter sensitivity interactively

> ⏱ Runs in **< 5 seconds**. No GPU. No MNIST download.


## 1 — Setup


In [ ]:
import json, pathlib, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

# ── Load results (relative path works from repo root or src/)
_here = pathlib.Path("results/raw_results.json")
if not _here.exists():
    _here = pathlib.Path("../results/raw_results.json")

with open(_here) as f:
    data = json.load(f)

with open(_here.with_name("verdicts.json")) as f:
    verdicts = json.load(f)

RHOS   = [r["rho"]      for r in data["low"]]
NOISES = ["low", "medium", "high"]
COLORS = {"low": "#2196F3", "medium": "#FF9800", "high": "#F44336"}
LABELS = {"low": "Low noise (5%)", "medium": "Medium noise (20%)", "high": "High noise (40%)"}

print("✅ Results loaded.")
print(f"   Noise levels : {NOISES}")
print(f"   ρ grid       : {RHOS}")
print(f"   Verdicts     : {verdicts}")


## 2 — The result, front and center

The key finding: **low noise has an interior peak at ρ=0.7; medium and high noise peak at the boundary ρ=1.0.**
That falsified P1, but P2 (the peak shifts) is clearly visible in the same data.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Connection-Density Trade-off  ·  MNIST Federated Learning  ·  v2.1",
             fontsize=13, fontweight="bold")

# ── Left: accuracy curves ──────────────────────────────────────
ax = axes[0]
for noise in NOISES:
    accs = [r["acc_mean"] for r in data[noise]]
    stds = [r["acc_std"]  for r in data[noise]]
    peak_idx = int(np.argmax(accs))
    ax.errorbar(RHOS, accs, yerr=stds,
                marker="o", linewidth=2, capsize=4,
                color=COLORS[noise], label=LABELS[noise])
    # Annotate the peak
    ax.annotate(
        f"ρ*={RHOS[peak_idx]}",
        xy=(RHOS[peak_idx], accs[peak_idx]),
        xytext=(RHOS[peak_idx] + 0.03, accs[peak_idx] + 0.005),
        fontsize=9, color=COLORS[noise], fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=COLORS[noise], lw=1.2),
    )

# Shade the "interior" region
ax.axvspan(0.1, 0.9, alpha=0.05, color="green", label="Interior region (0<ρ<1)")
ax.axvline(1.0, color="grey", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Sync density ρ", fontsize=11)
ax.set_ylabel("Test accuracy", fontsize=11)
ax.set_title("Accuracy vs. Sync Density\n(3 noise levels, 3 seeds each)", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0.05, 1.08)

# ── Right: comm cost (linear proof) ───────────────────────────
ax2 = axes[1]
comm = [r["comm"] for r in data["low"]]
ax2.plot(RHOS, comm, marker="s", linewidth=2.5,
         color="#9C27B0", label="Comm cost (proxy)")
ax2.plot(RHOS, [a * 100 for a in [r["acc_mean"] for r in data["low"]]],
         marker="^", linewidth=2, linestyle="--",
         color="#4CAF50", label="Accuracy × 100 (low noise)")
ax2.set_xlabel("Sync density ρ", fontsize=11)
ax2.set_ylabel("Value", fontsize=11)
ax2.set_title("P4: Comm Cost Linear ↑,\nAccuracy Gain Sub-linear", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# Callout box
ax2.annotate("10× comm cost\n→ only 1.17× accuracy",
             xy=(1.0, 100), xytext=(0.4, 95),
             fontsize=9, color="#9C27B0", fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.3", fc="lavender", alpha=0.8),
             arrowprops=dict(arrowstyle="->", color="#9C27B0"))

plt.tight_layout()
plt.savefig("results/annotated_results.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Annotated plot saved to results/annotated_results.png")


## 3 — The Story: Hypothesis → Experiment → Failure → Insight


In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class StoryStep:
    stage: str
    summary: str
    detail: str
    status: str  # "✅" | "❌" | "⚠️"

story = [
    StoryStep(
        stage="1 · HYPOTHESIS",
        summary="Multi-agent systems have a trade-off: consensus ↑ accuracy, sharing ↓ diversity.",
        detail=(
            "Model: J(ρ) = V(ρ) + η·ρᵖ\n"
            "Prior simulation (36 regimes) showed ρ* ranges 0.10–1.00.\n"
            "ρ* = 0.5 occurs in only ~6% of cases → NOT a universal constant."
        ),
        status="✅",
    ),
    StoryStep(
        stage="2 · PRE-REGISTRATION",
        summary="Four predictions locked before ANY real data was seen.",
        detail=(
            "P1: Interior optimum ρ* ∈ (0,1) at every noise level\n"
            "P2: ρ* shifts with noise (more noise → higher ρ*)\n"
            "P3: ρ* ≠ 0.5 universally\n"
            "P4: Comm cost linear; accuracy gain sub-linear"
        ),
        status="✅",
    ),
    StoryStep(
        stage="3 · EXPERIMENT",
        summary="MNIST federated learning: 10 clients, 3 noise levels, 5 ρ values, 3 seeds.",
        detail="Code: src/fed_experiment.py · Runtime: ~8 min CPU · Seed: 42",
        status="✅",
    ),
    StoryStep(
        stage="4 · FAILURE",
        summary="P1 failed. Interior optimum appeared at low noise only.",
        detail=(
            "low noise:    peak at ρ*=0.7  ✅ interior\n"
            "medium noise: peak at ρ*=1.0  ❌ boundary\n"
            "high noise:   peak at ρ*=1.0  ❌ boundary\n\n"
            "Root cause: MNIST clients all draw from same distribution → β/γ ≫ 1\n"
            "The sensitivity analysis already predicted ρ* → 1.0 in this regime."
        ),
        status="❌",
    ),
    StoryStep(
        stage="5 · INSIGHT",
        summary="The framework was right; the prediction was too strong.",
        detail=(
            "Revised claim: 'An interior optimum may exist when β/γ is moderate.'\n"
            "Practical rule: 10× bandwidth buys only 1.17× accuracy — measure first.\n"
            "Next test: non-IID splits (each client sees only 2 digit classes).\n"
            "If interior optimum reappears there, P1 is rehabilitated under scoped form."
        ),
        status="⭐",
    ),
]

# Print the narrative
width = 72
for step in story:
    print(f"\n{'━' * width}")
    print(f"  {step.status}  {step.stage}")
    print(f"{'━' * width}")
    print(f"  {step.summary}")
    print()
    for line in step.detail.splitlines():
        print(f"    {line}")
print(f"\n{'━' * width}")


## 4 — P1–P4 Verdict Walkthrough (step-by-step logic)


In [ ]:
def get_peak(noise_name):
    accs = [r["acc_mean"] for r in data[noise_name]]
    idx  = int(np.argmax(accs))
    return RHOS[idx], idx, accs[idx]

print("─" * 60)
print("P1 — Interior optimum ρ* ∈ (0,1) at every noise level")
print("─" * 60)
p1_results = {}
for noise in NOISES:
    rho_star, idx, acc = get_peak(noise)
    interior = 0 < idx < len(RHOS) - 1
    symbol = "✅ INTERIOR" if interior else "❌ BOUNDARY"
    print(f"  {noise:8s}: ρ*={rho_star}  acc={acc:.4f}  → {symbol}")
    p1_results[noise] = interior

p1_pass = all(p1_results.values())
print(f"\n  P1 verdict: {'✅ PASS' if p1_pass else '❌ FAIL'}")

print("\n─" * 61)
print("P2 — ρ* shifts with noise level")
print("─" * 60)
peak_rhos = [get_peak(n)[0] for n in NOISES]
print(f"  Peaks across noise levels: {dict(zip(NOISES, peak_rhos))}")
p2_pass = len(set(peak_rhos)) > 1
print(f"  All same? {not p2_pass}  →  P2 verdict: {'✅ PASS' if p2_pass else '❌ FAIL'}")

print("\n─" * 61)
print("P3 — ρ* ≠ 0.5 universally")
print("─" * 60)
any_half = any(r == 0.5 for r in peak_rhos)
print(f"  Any peak at 0.5? {any_half}  →  P3 verdict: {'❌ FAIL' if any_half else '✅ PASS'}")

print("\n─" * 61)
print("P4 — Comm cost linear, accuracy sub-linear")
print("─" * 60)
accs_low  = [r["acc_mean"] for r in data["low"]]
comms_low = [r["comm"]     for r in data["low"]]
acc_ratio  = accs_low[-1]  / accs_low[0]
comm_ratio = comms_low[-1] / comms_low[0]
print(f"  Accuracy gain (ρ=0.1→1.0) : {acc_ratio:.2f}×")
print(f"  Comm cost gain             : {comm_ratio:.2f}×")
p4_pass = acc_ratio < comm_ratio
print(f"  Sub-linear? {p4_pass}  →  P4 verdict: {'✅ PASS' if p4_pass else '❌ FAIL'}")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for pid, verdict in verdicts.items():
    sym = "✅" if "PASS" in verdict else "❌"
    print(f"  {sym} {pid}: {verdict}")


## 5 — Interactive Explorer

Use the dropdown to isolate a noise level and the slider to mark a custom ρ threshold.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ℹ️  ipywidgets not installed — run:  pip install ipywidgets")
    print("   Falling back to static plot.\n")

def plot_noise_level(noise_name="low", rho_threshold=0.5):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    rows = data[noise_name]
    accs = [r["acc_mean"] for r in rows]
    stds = [r["acc_std"]  for r in rows]
    peak_idx = int(np.argmax(accs))

    ax.fill_between(RHOS, [a - s for a, s in zip(accs, stds)],
                         [a + s for a, s in zip(accs, stds)],
                    alpha=0.15, color=COLORS[noise_name])
    ax.plot(RHOS, accs, marker="o", linewidth=2.5,
            color=COLORS[noise_name], label=LABELS[noise_name])

    # Peak marker
    ax.scatter([RHOS[peak_idx]], [accs[peak_idx]],
               s=140, zorder=5, color=COLORS[noise_name],
               edgecolor="black", linewidth=1.5, label=f"Peak ρ*={RHOS[peak_idx]}")

    # ρ threshold line
    ax.axvline(rho_threshold, color="grey", linestyle="--", linewidth=1.2,
               label=f"Your threshold ρ={rho_threshold:.1f}")

    # Interior/boundary call
    is_interior = 0 < peak_idx < len(RHOS) - 1
    verdict_txt = ("✅ Interior optimum" if is_interior else "❌ Boundary peak (full sync wins)")
    ax.set_title(f"{LABELS[noise_name]}  ·  {verdict_txt}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Sync density ρ"); ax.set_ylabel("Test accuracy")
    ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(0.05, 1.08)
    ax.set_ylim(0.60, 0.92)
    plt.tight_layout(); plt.show()

if HAS_WIDGETS:
    noise_dd = widgets.Dropdown(
        options=["low", "medium", "high"],
        value="low", description="Noise level:"
    )
    rho_slider = widgets.FloatSlider(
        value=0.5, min=0.1, max=1.0, step=0.1,
        description="ρ threshold:", readout_format=".1f"
    )
    ui = widgets.HBox([noise_dd, rho_slider])
    out = widgets.interactive_output(
        plot_noise_level,
        {"noise_name": noise_dd, "rho_threshold": rho_slider}
    )
    display(ui, out)
else:
    # Static fallback — show all three
    for n in NOISES:
        plot_noise_level(n)


## 6 — Practical Regime Guide

Given your β/γ ratio, which ρ should you use?


In [ ]:
def recommend_rho(beta_over_gamma: float) -> dict:
    """
    Translate β/γ ratio into a practical ρ recommendation,
    grounded in this experiment's findings.
    """
    if beta_over_gamma > 2.0:
        rec = "ρ = 1.0 (full sync)"
        reason = "β/γ ≫ 1 → consensus benefit dominates. Full sync wins. Don't sweep."
        confidence = "High — validated empirically at medium & high noise."
    elif beta_over_gamma >= 0.5:
        rec = "Run a ρ sweep ∈ {0.5, 0.7, 1.0}"
        reason = "β/γ ~ 1 → moderate regime. Interior optimum likely. Measure it."
        confidence = "Medium — only low-noise condition validated."
    else:
        rec = "ρ ≈ 0.2–0.4 (sparse sync)"
        reason = "β/γ ≪ 1 → diversity cost dominates. Sparse sync wins."
        confidence = "Low — not directly tested in this experiment."
    return {"β/γ": beta_over_gamma, "recommendation": rec,
            "reason": reason, "confidence": confidence}

# Print a guide table
print(f"{'β/γ':>8}  {'Recommendation':<30}  Confidence")
print("─" * 75)
for ratio in [0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]:
    r = recommend_rho(ratio)
    print(f"{ratio:>8.1f}  {r['recommendation']:<30}  {r['confidence'].split('—')[0].strip()}")

print("\n─" * 76)
print("This experiment ran in the β/γ ≫ 1 regime → full sync dominated (P1 failed).")
print("To test the interior regime: use non-IID client splits (next experiment).")
